In [1]:
from google.colab import drive

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import zipfile
from pathlib import Path

# Set your zip file path in Drive
zip_path = Path("/content/drive/MyDrive/colab/mlpr/data.zip")

# Extract into the same folder as the zip file
extract_dir = zip_path.parent
with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall(extract_dir)

print(f"Extracted '{zip_path.name}' to '{extract_dir}'")

Extracted 'data.zip' to '/content/drive/MyDrive/colab/mlpr'


In [5]:
import pandas as pd
data_dir = "/content/drive/MyDrive/colab/mlpr/data"
station_data = pd.read_csv("/content/drive/MyDrive/colab/mlpr/stations_master.csv")

In [ ]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
from difflib import SequenceMatcher

# Change this base path if needed
base_dir = Path("/content/drive/MyDrive/colab/mlpr")
data_dir = base_dir / "data"
stations_master_path = base_dir / "stations_master.csv"
output_path = base_dir / "merged_data.csv"

stations = pd.read_csv(stations_master_path)

def norm_text(s):
    s = str(s).lower().strip()
    s = s.replace("_", " ").replace("-", " ")
    s = re.sub(r"[^\w\s,]", " ", s)     # remove special chars, keep comma
    s = re.sub(r"\s+", " ", s).strip()
    return s

def station_core(s):
    # Keep the most identifying part before comma/agency suffix
    s = norm_text(s)
    s = s.split(",")[0].strip()          # before city part
    return s

def norm_state_city(s):
    return norm_text(s).replace(" ", "_")

# Precompute normalized metadata
meta_rows = []
for _, r in stations.iterrows():
    meta_rows.append({
        "state_raw": r["state"],
        "city_raw": r["city"],
        "station_raw": r["station_name"],
        "state_norm": norm_state_city(r["state"]),
        "city_norm": norm_text(r["city"]),
        "station_norm": norm_text(r["station_name"]),
        "station_core": station_core(r["station_name"]),
        "latitude": r["latitude"],
        "longitude": r["longitude"],
    })

def similarity(a, b):
    return SequenceMatcher(None, a, b).ratio()

def best_station_match(folder_state, folder_city, folder_station, min_score=0.60):
    fs = norm_state_city(folder_state)
    fc = norm_text(folder_city)
    fstation_norm = norm_text(folder_station)
    fstation_core = station_core(folder_station)

    # 1) Prefer candidates from same state + city
    candidates = [m for m in meta_rows if m["state_norm"] == fs and m["city_norm"] == fc]

    # 2) Fallback to same state if city text differs
    if not candidates:
        candidates = [m for m in meta_rows if m["state_norm"] == fs]

    # 3) Last fallback: all rows
    if not candidates:
        candidates = meta_rows

    best = None
    best_score = -1.0

    for m in candidates:
        # combine full-name and core-name similarity
        s1 = similarity(fstation_norm, m["station_norm"])
        s2 = similarity(fstation_core, m["station_core"])
        score = 0.45 * s1 + 0.55 * s2

        if score > best_score:
            best_score = score
            best = m

    if best is not None and best_score >= min_score:
        return best, best_score
    return None, best_score

# List all immediate subfolders inside data_dir
states = sorted([p for p in data_dir.iterdir() if p.is_dir()])

rows = []

for state in states:
    cities = sorted([p for p in state.iterdir() if p.is_dir()])
    for city in cities:
        station_dirs = sorted([p for p in city.iterdir() if p.is_dir()])

        for station in station_dirs:
            match, score = best_station_match(state.name, city.name, station.name, min_score=0.30)
            if not match:
                print(f"No good match for '{station.name}' (score: {score:.2f})")
            else:
                print(f"Matched '{station.name}' to '{match['station_raw']}' (score: {score:.2f})")
                rows.append({
                    "state": state.name,
                    "city": city.name,
                    "station_name": station.name,
                    "latitude": match["latitude"],
                    "longitude": match["longitude"],
                })

df = pd.DataFrame(rows, columns=["state", "city", "station_name", "latitude", "longitude"])

df.to_csv(output_path, index=False)

Matched 'Police Line, Sri Vijaya Puram - ANPCC' to 'Police Line, Saharsa - BSPCB' (score: 0.83)
Matched 'Secretariat, Amaravati - APPCB' to 'Secretariat, Amaravati - APPCB' (score: 1.00)
Matched 'Gulzarpet, Anantapur - APPCB' to 'Gulzarpet, Anantapur - APPCB' (score: 1.00)
Matched 'Gangineni Cheruvu, Chittoor - APPCB' to 'Gangineni Cheruvu, Chittoor - APPCB' (score: 1.00)
Matched 'District Court, Eluru - APPCB' to 'District Court, Eluru - APPCB' (score: 1.00)
Matched 'Rajendra Nagar North, Guntur - APPCB' to 'Rajendra Nagar North, Guntur - APPCB' (score: 1.00)
Matched 'Yerramukkapalli, Kadapa - APPCB' to 'Yerramukkapalli, Kadapa - APPCB' (score: 1.00)
Matched 'Srinivas Nagar Colony, Machilipatnam - APPCB' to 'Srinivas Nagar Colony, Machilipatnam - APPCB' (score: 1.00)
Matched 'Anand Kala Kshetram, Rajamahendravaram - APPCB' to 'Anand Kala Kshetram, Rajamahendravaram - APPCB' (score: 1.00)
Matched 'Toll Gate, Tirumala - APPCB' to 'Toll Gate, Tirumala - APPCB' (score: 1.00)
Matched 'Vaik

In [7]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
from difflib import SequenceMatcher

# Configuration
base_dir = Path("/content/drive/MyDrive/colab/mlpr")
data_dir = base_dir / "data"
stations_master_path = base_dir / "stations_master.csv"
output_dir = base_dir / "merged_states"  # Output folder for individual state CSVs

# Create output directory
output_dir.mkdir(exist_ok=True)

# States to process
states_to_process = [
#   "Andaman and Nicobar",
#   "Andhra Pradesh",
#   "Arunachal Pradesh",
#   "Assam",
#   "Bihar",
# "Chandigarh",
#   "Chhattisgarh",
# "Delhi",
#   "Gujarat",
#   "Haryana",
#   "Himachal Pradesh",
#   "Jammu and Kashmir",
#   "Jharkhand",
#   "Karnataka",
#   "Kerala",
#   "Madhya Pradesh",
  "Maharashtra",
  "Manipur",
  "Meghalaya",
  "Mizoram",
  "Nagaland",
  "Odisha",
  "Puducherry",
  "Punjab",
  "Rajasthan",
  "Sikkim",
  "Tamil Nadu",
  "Telangana",
  "Tripura",
  "Uttar Pradesh",
  "Uttarakhand",
  "West Bengal",
];


# Read station metadata once
stations = pd.read_csv(stations_master_path)

def norm_text(s):
    s = str(s).lower().strip()
    s = s.replace("_", " ").replace("-", " ")
    s = re.sub(r"[^\w\s,]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def station_core(s):
    s = norm_text(s)
    s = s.split(",")[0].strip()
    return s

def norm_state_city(s):
    return norm_text(s).replace(" ", "_")

# Precompute normalized metadata
meta_rows = []
for _, r in stations.iterrows():
    meta_rows.append({
        "state_raw": r["state"],
        "city_raw": r["city"],
        "station_raw": r["station_name"],
        "state_norm": norm_state_city(r["state"]),
        "city_norm": norm_text(r["city"]),
        "station_norm": norm_text(r["station_name"]),
        "station_core": station_core(r["station_name"]),
        "latitude": r["latitude"],
        "longitude": r["longitude"],
    })

def similarity(a, b):
    return SequenceMatcher(None, a, b).ratio()

def best_station_match(folder_state, folder_city, folder_station, min_score=0.60):
    fs = norm_state_city(folder_state)
    fc = norm_text(folder_city)
    fstation_norm = norm_text(folder_station)
    fstation_core = station_core(folder_station)

    # Prefer same state + city
    candidates = [m for m in meta_rows if m["state_norm"] == fs and m["city_norm"] == fc]
    if not candidates:
        candidates = [m for m in meta_rows if m["state_norm"] == fs]
    if not candidates:
        candidates = meta_rows

    best = None
    best_score = -1.0

    for m in candidates:
        s1 = similarity(fstation_norm, m["station_norm"])
        s2 = similarity(fstation_core, m["station_core"])
        score = 0.45 * s1 + 0.55 * s2

        if score > best_score:
            best_score = score
            best = m

    if best is not None and best_score >= min_score:
        return best, best_score
    return None, best_score

# Process each state
for state_name in states_to_process:
    print(f"\n{'='*60}")
    print(f"Processing: {state_name}")
    print('='*60)
    
    state_path = data_dir / state_name
    if not state_path.exists():
        print(f"  ✗ State folder not found: {state_path}")
        continue
    
    all_frames = []
    unmatched = []
    
    # Collect all CSV files in this state
    csv_files = sorted(state_path.glob("*/*/*.csv"))  # city/station/year.csv
    print(f"  Found {len(csv_files)} CSV files")
    
    for i, csv_path in enumerate(csv_files, 1):
        if i % 100 == 0:
            print(f"    Processing file {i}/{len(csv_files)}...")
        
        rel = csv_path.relative_to(state_path)
        city, station_folder, year_file = rel.parts
        year = Path(year_file).stem
        
        try:
            df = pd.read_csv(csv_path)
        except Exception as e:
            print(f"    ✗ Error reading {csv_path.name}: {e}")
            continue
        
        match, score = best_station_match(state_name, city, station_folder, min_score=0.30)
        
        # Add metadata columns
        df["state"] = state_name
        df["city"] = city
        df["station_name"] = station_folder
        df["year"] = year
        
        if match is not None:
            df["latitude"] = match["latitude"]
            df["longitude"] = match["longitude"]
        else:
            df["latitude"] = np.nan
            df["longitude"] = np.nan
            unmatched.append((city, station_folder, score))
        
        all_frames.append(df)
        
        # Save in smaller batches to control memory
        if len(all_frames) >= 50:
            print(f"    Flushing batch of {len(all_frames)} files...")
            batch_df = pd.concat(all_frames, ignore_index=True)
            all_frames = []
            
            # Save to temporary intermediate file (or directly append)
            # For now, keep in memory up to final concatenation
            # To truly limit memory, you'd append rows to CSV incrementally
    
    # Final merge for this state
    if all_frames:
        state_df = pd.concat(all_frames, ignore_index=True)
        
        # Reorder columns
        added_cols = ["state", "city", "station_name", "latitude", "longitude", "year"]
        base_cols = [c for c in state_df.columns if c not in added_cols]
        state_df = state_df[base_cols + added_cols]
        
        # Save to state-specific output file
        output_file = output_dir / f"merged_{state_name.replace(' ', '_')}.csv"
        state_df.to_csv(output_file, index=False)
        
        print(f"\n  ✓ Merged {len(state_df)} rows")
        print(f"  ✓ Saved: {output_file.name}")
        print(f"  Columns: {state_df.shape[1]}")
        
        if unmatched:
            print(f"\n  ⚠ Unmatched stations: {len(unmatched)}")
            for city, station, score in unmatched[:10]:
                print(f"    - {city} / {station} (score: {score:.2f})")
    else:
        print(f"  ✗ No data processed")

print(f"\n{'='*60}")
print("All states processed!")
print(f"Output folder: {output_dir}")


Processing: Maharashtra
  Found 405 CSV files
    Flushing batch of 50 files...
    Processing file 100/405...
    Flushing batch of 50 files...
    Flushing batch of 50 files...
    Processing file 200/405...
    Flushing batch of 50 files...
    Flushing batch of 50 files...
    Processing file 300/405...
    Flushing batch of 50 files...
    Flushing batch of 50 files...


/tmp/ipykernel_34532/2140634448.py:171: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  batch_df = pd.concat(all_frames, ignore_index=True)


    Processing file 400/405...
    Flushing batch of 50 files...


/tmp/ipykernel_34532/2140634448.py:171: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  batch_df = pd.concat(all_frames, ignore_index=True)



  ✓ Merged 43824 rows
  ✓ Saved: merged_Maharashtra.csv
  Columns: 31

Processing: Manipur
  Found 12 CSV files

  ✓ Merged 105216 rows
  ✓ Saved: merged_Manipur.csv
  Columns: 31

Processing: Meghalaya
  Found 12 CSV files

  ✓ Merged 105216 rows
  ✓ Saved: merged_Meghalaya.csv
  Columns: 31

Processing: Mizoram
  Found 6 CSV files

  ✓ Merged 52608 rows
  ✓ Saved: merged_Mizoram.csv
  Columns: 31

Processing: Nagaland
  Found 1 CSV files

  ✓ Merged 8760 rows
  ✓ Saved: merged_Nagaland.csv
  Columns: 31

Processing: Odisha
  Found 73 CSV files
    Flushing batch of 50 files...


...

    Processing file 200/340...
    Flushing batch of 50 files...
    Flushing batch of 50 files...
    Processing file 300/340...
    Flushing batch of 50 files...


/tmp/ipykernel_34532/2140634448.py:180: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  state_df = pd.concat(all_frames, ignore_index=True)



  ✓ Merged 341952 rows
  ✓ Saved: merged_Uttar_Pradesh.csv
  Columns: 31

Processing: Uttarakhand
  Found 12 CSV files


/tmp/ipykernel_34532/2140634448.py:180: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  state_df = pd.concat(all_frames, ignore_index=True)



  ✓ Merged 86256 rows
  ✓ Saved: merged_Uttarakhand.csv
  Columns: 31

Processing: West Bengal
  Found 93 CSV files
    Flushing batch of 50 files...

  ✓ Merged 377016 rows
  ✓ Saved: merged_West_Bengal.csv
  Columns: 31

All states processed!
Output folder: /content/drive/MyDrive/colab/mlpr/merged_states


In [14]:
import shutil
from google.colab import files

!ls /content/drive/MyDrive/colab/mlpr/merged_states
zip_file = shutil.make_archive(str(output_dir), "zip", root_dir=str(output_dir))
files.download(zip_file)

merged_Andaman_and_Nicobar.csv	merged_Maharashtra.csv
merged_Andhra_Pradesh.csv	merged_Manipur.csv
merged_Arunachal_Pradesh.csv	merged_Meghalaya.csv
merged_Assam.csv		merged_Mizoram.csv
merged_Bihar.csv		merged_Nagaland.csv
merged_Chandigarh.csv		merged_Odisha.csv
merged_Chhattisgarh.csv		merged_Puducherry.csv
merged_Delhi.csv		merged_Punjab.csv
merged_Gujarat.csv		merged_Rajasthan.csv
merged_Haryana.csv		merged_Sikkim.csv
merged_Himachal_Pradesh.csv	merged_Tamil_Nadu.csv
merged_Jammu_and_Kashmir.csv	merged_Telangana.csv
merged_Jharkhand.csv		merged_Tripura.csv
merged_Karnataka.csv		merged_Uttarakhand.csv
merged_Kerala.csv		merged_Uttar_Pradesh.csv
merged_Madhya_Pradesh.csv	merged_West_Bengal.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [28]:
df = pd.read_csv(output_dir / "merged_Maharashtra.csv")
df["NOx (ppb)"].isna().sum()

np.int64(573602)

In [27]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
from difflib import SequenceMatcher

# Configuration
base_dir = Path("/content/drive/MyDrive/colab/mlpr")
data_dir = base_dir / "data"
stations_master_path = base_dir / "stations_master.csv"
output_dir = base_dir / "merged_states"  # Output folder for individual state CSVs

# Create output directory
output_dir.mkdir(exist_ok=True)

# States to process
states_to_process = [
#   "Andaman and Nicobar",
#   "Andhra Pradesh",
#   "Arunachal Pradesh",
#   "Assam",
#   "Bihar",
# "Chandigarh",
#   "Chhattisgarh",
# "Delhi",
#   "Gujarat",
#   "Haryana",
#   "Himachal Pradesh",
#   "Jammu and Kashmir",
#   "Jharkhand",
#   "Karnataka",
#   "Kerala",
#   "Madhya Pradesh",
  "Maharashtra",
#   "Manipur",
#   "Meghalaya",
#   "Mizoram",
#   "Nagaland",
#   "Odisha",
#   "Puducherry",
#   "Punjab",
#   "Rajasthan",
#   "Sikkim",
#   "Tamil Nadu",
#   "Telangana",
#   "Tripura",
#   "Uttar Pradesh",
#   "Uttarakhand",
#   "West Bengal",
];


# Read station metadata once
stations = pd.read_csv(stations_master_path)

def norm_text(s):
    s = str(s).lower().strip()
    s = s.replace("_", " ").replace("-", " ")
    s = re.sub(r"[^\w\s,]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def station_core(s):
    s = norm_text(s)
    s = s.split(",")[0].strip()
    return s

def norm_state_city(s):
    return norm_text(s).replace(" ", "_")

# Precompute normalized metadata
meta_rows = []
for _, r in stations.iterrows():
    meta_rows.append({
        "state_raw": r["state"],
        "city_raw": r["city"],
        "station_raw": r["station_name"],
        "state_norm": norm_state_city(r["state"]),
        "city_norm": norm_text(r["city"]),
        "station_norm": norm_text(r["station_name"]),
        "station_core": station_core(r["station_name"]),
        "latitude": r["latitude"],
        "longitude": r["longitude"],
    })

def similarity(a, b):
    return SequenceMatcher(None, a, b).ratio()

def best_station_match(folder_state, folder_city, folder_station, min_score=0.60):
    fs = norm_state_city(folder_state)
    fc = norm_text(folder_city)
    fstation_norm = norm_text(folder_station)
    fstation_core = station_core(folder_station)

    # Prefer same state + city
    candidates = [m for m in meta_rows if m["state_norm"] == fs and m["city_norm"] == fc]
    if not candidates:
        candidates = [m for m in meta_rows if m["state_norm"] == fs]
    if not candidates:
        candidates = meta_rows

    best = None
    best_score = -1.0

    for m in candidates:
        s1 = similarity(fstation_norm, m["station_norm"])
        s2 = similarity(fstation_core, m["station_core"])
        score = 0.45 * s1 + 0.55 * s2

        if score > best_score:
            best_score = score
            best = m

    if best is not None and best_score >= min_score:
        return best, best_score
    return None, best_score

# Process each state
for state_name in states_to_process:
    print(f"\n{'='*60}")
    print(f"Processing: {state_name}")
    print('='*60)
    
    state_path = data_dir / state_name
    if not state_path.exists():
        print(f"  ✗ State folder not found: {state_path}")
        continue
    
    all_frames = []
    unmatched = []
    
    # Collect all CSV files in this state
    csv_files = sorted(state_path.glob("*/*/*.csv"))  # city/station/year.csv
    print(f"  Found {len(csv_files)} CSV files")
    
    for i, csv_path in enumerate(csv_files, 1):
        if i % 100 == 0:
            print(f"    Processing file {i}/{len(csv_files)}...")
        
        rel = csv_path.relative_to(state_path)
        city, station_folder, year_file = rel.parts
        year = Path(year_file).stem

        if year in ["2020", "2021", "2022"]:
            continue
        
        try:
            df = pd.read_csv(csv_path)
        except Exception as e:
            print(f"    ✗ Error reading {csv_path.name}: {e}")
            continue
        
        match, score = best_station_match(state_name, city, station_folder, min_score=0.30)
        
        # Add metadata columns
        df["state"] = state_name
        df["city"] = city
        df["station_name"] = station_folder
        df["year"] = year
        
        if match is not None:
            df["latitude"] = match["latitude"]
            df["longitude"] = match["longitude"]
        else:
            df["latitude"] = np.nan
            df["longitude"] = np.nan
            unmatched.append((city, station_folder, score))

        # Data cleaning
        df.columns = df.columns.str.strip()

        if "Timestamp" in df.columns:
            df["Timestamp"] = pd.to_datetime(df["Timestamp"], errors="coerce")
            df = df.dropna(subset=["Timestamp"]).sort_values("Timestamp")
            df = df.drop_duplicates(subset=["Timestamp"], keep="last")

        meta_cols = {"Timestamp", "state", "city", "station_name", "year", "latitude", "longitude"}
        numeric_cols = [c for c in df.columns if c not in meta_cols]

        for c in numeric_cols:
            df[c] = pd.to_numeric(df[c], errors="coerce")

        df[numeric_cols] = df[numeric_cols].replace([np.inf, -np.inf], np.nan)

        # Physical validity checks
        non_negative_cols = [c for c in numeric_cols if c not in {"AT (°C)", "WD (deg)"}]
        if non_negative_cols:
            df[non_negative_cols] = df[non_negative_cols].mask(df[non_negative_cols] < 0)

        if "RH (%)" in df.columns:
            df["RH (%)"] = df["RH (%)"].where(df["RH (%)"].between(0, 100), np.nan)

        if "WD (deg)" in df.columns:
            df["WD (deg)"] = df["WD (deg)"].where(df["WD (deg)"].between(0, 360), np.nan)

        # Drop rows where all measurements are missing
        if numeric_cols:
            bad_count = (df[numeric_cols].isna() | (df[numeric_cols] == 0)).sum(axis=1)
            df = df.loc[~bad_count.between(3, 4)].reset_index(drop=True)
            df = df.dropna(subset=numeric_cols, how="all").reset_index(drop=True)
        

        
        all_frames.append(df)
    
    # Final merge for this state
    if all_frames:
        state_df = pd.concat(all_frames, ignore_index=True)
        
        # Reorder columns
        added_cols = ["state", "city", "station_name", "latitude", "longitude", "year"]
        base_cols = [c for c in state_df.columns if c not in added_cols]
        state_df = state_df[base_cols + added_cols]
        
        # Save to state-specific output file
        output_file = output_dir / f"merged_{state_name.replace(' ', '_')}.csv"
        state_df.to_csv(output_file, index=False)
        
        print(f"\n  ✓ Merged {len(state_df)} rows")
        print(f"  ✓ Saved: {output_file.name}")
        print(f"  Columns: {state_df.shape[1]}")
        
        if unmatched:
            print(f"\n  ⚠ Unmatched stations: {len(unmatched)}")
            for city, station, score in unmatched[:10]:
                print(f"    - {city} / {station} (score: {score:.2f})")
    else:
        print(f"  ✗ No data processed")

print(f"\n{'='*60}")
print("All states processed!")
print(f"Output folder: {output_dir}")


Processing: Maharashtra
  Found 405 CSV files
    Processing file 100/405...
    Processing file 200/405...
    Processing file 300/405...
    Processing file 400/405...

  ✓ Merged 2382705 rows
  ✓ Saved: merged_Maharashtra.csv
  Columns: 31

All states processed!
Output folder: /content/drive/MyDrive/colab/mlpr/merged_states


In [26]:
df = pd.read_csv('/content/drive/MyDrive/colab/mlpr/final_dataset.csv')
print(df.shape)

(887061, 40)
